# 00-01 环境搭建与工具链配置

**目标**: 搭建完整的学习环境，确保所有工具链可用。

**完成标志**:
- [x] Python 虚拟环境就绪
- [x] 所有依赖安装成功
- [x] API Keys 配置正确
- [x] 能成功调用至少一个 LLM API

---

## 1. 环境准备

### 1.1 Python 版本检查
本项目要求 Python >= 3.10（推荐 3.11+），因为 LangGraph、AutoGen 等框架依赖较新的 Python 特性。

In [ ]:
import sys
print(f"Python 版本: {sys.version}")
assert sys.version_info >= (3, 10), "需要 Python 3.10+，请升级！"
print("✓ Python 版本符合要求")

### 1.2 安装依赖

如果还未安装，运行以下命令：
```bash
pip install -r ../requirements.txt
```

In [ ]:
# 验证核心依赖是否安装成功
import importlib

core_packages = [
    ("numpy", "NumPy"),
    ("pandas", "Pandas"),
    ("pydantic", "Pydantic"),
    ("openai", "OpenAI SDK"),
    ("anthropic", "Anthropic SDK"),
    ("langchain", "LangChain"),
    ("langgraph", "LangGraph"),
    ("chromadb", "ChromaDB"),
    ("transformers", "HuggingFace Transformers"),
    ("peft", "PEFT (LoRA)"),
    ("duckdb", "DuckDB"),
]

for pkg_name, display_name in core_packages:
    try:
        mod = importlib.import_module(pkg_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  ✓ {display_name}: {version}")
    except ImportError:
        print(f"  ✗ {display_name}: 未安装! 请运行 pip install {pkg_name}")

## 2. API Keys 配置

我们使用 `.env` 文件管理 API Keys，**绝对不要**将真实密钥提交到 Git。

### 步骤:
1. 复制 `.env.example` 为 `.env`
2. 填入你的 API Keys
3. `.gitignore` 已配置忽略 `.env`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 加载 .env 文件
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(env_path)
    print("✓ .env 文件已加载")
else:
    print("⚠ .env 文件不存在！请复制 .env.example 为 .env 并填入 API Keys")
    print("  运行: cp ../.env.example ../.env")

# 检查关键 API Keys (不打印实际值)
keys_to_check = [
    ("OPENAI_API_KEY", "OpenAI"),
    ("ANTHROPIC_API_KEY", "Anthropic (Claude)"),
    ("DASHSCOPE_API_KEY", "通义千问"),
    ("LANGCHAIN_API_KEY", "LangSmith"),
    ("HF_TOKEN", "HuggingFace"),
]

for key_name, display_name in keys_to_check:
    value = os.getenv(key_name)
    if value and not value.startswith("sk-your") and not value.startswith("lsv2_your") and not value.startswith("hf_your"):
        print(f"  ✓ {display_name}: 已配置 ({key_name[:4]}...{key_name[-4:]})")
    else:
        print(f"  ○ {display_name}: 未配置 (可选，用到时再配)")

## 3. LLM API 调用测试

验证至少一个 LLM API 可以正常工作。我们使用 `utils/llm_client.py` 中的统一封装。

### 3.1 使用 OpenAI API

In [ ]:
# 添加项目根目录到 path
import sys
sys.path.insert(0, "..")

from utils.llm_client import call_llm

# 测试调用 (如果你配置了 OpenAI key)
try:
    response = call_llm(
        prompt="用一句话介绍什么是 AI Agent",
        provider="openai",
        model="gpt-4o-mini"
    )
    print(f"OpenAI 响应:\n{response}")
except Exception as e:
    print(f"OpenAI 调用失败 (可能未配置 key): {e}")

### 3.2 使用 Anthropic Claude API

In [ ]:
try:
    response = call_llm(
        prompt="用一句话介绍什么是 RAG (Retrieval Augmented Generation)",
        provider="anthropic",
        model="claude-sonnet-4-20250514"
    )
    print(f"Claude 响应:\n{response}")
except Exception as e:
    print(f"Claude 调用失败 (可能未配置 key): {e}")

## 4. DuckDB 测试

DuckDB 是我们练习 SQL 的主力工具 — 无需安装数据库服务器，直接在 Notebook 中运行。

In [ ]:
import duckdb

# 创建内存数据库并测试
con = duckdb.connect()
result = con.sql("""
    SELECT 
        '环境搭建' AS module,
        'DuckDB 正常工作' AS status,
        current_timestamp AS check_time
""").fetchdf()

print(result)
print("\n✓ DuckDB 工作正常")
con.close()

## 5. 向量数据库测试

测试 ChromaDB (RAG 模块会用到)。

In [ ]:
import chromadb

# 创建内存向量库
client = chromadb.Client()
collection = client.create_collection(name="test")

# 插入测试数据
collection.add(
    documents=["AI Agent 是能自主完成任务的智能体", "RAG 是检索增强生成技术"],
    ids=["doc1", "doc2"]
)

# 查询
results = collection.query(query_texts=["什么是智能体"], n_results=1)
print(f"查询结果: {results['documents']}")
print("\n✓ ChromaDB 工作正常")

## 6. 环境检查总结

运行下面的总检查，确认环境就绪：

In [ ]:
print("=" * 50)
print("  环境搭建检查清单")
print("=" * 50)
checks = [
    ("Python >= 3.10", sys.version_info >= (3, 10)),
    (".env 文件存在", Path("../.env").exists()),
    ("核心依赖已安装", True),  # 如果跑到这说明前面没报错
    ("DuckDB 可用", True),
    ("ChromaDB 可用", True),
]

all_ok = True
for name, passed in checks:
    status = "✓" if passed else "✗"
    print(f"  {status} {name}")
    if not passed:
        all_ok = False

print("=" * 50)
if all_ok:
    print("  环境就绪！可以开始学习了 🚀")
else:
    print("  部分检查未通过，请按上面的提示修复")

---

## 下一步

环境搭建完成后，进入 **01-python-advanced** 开始 Python 进阶学习：
- `01_async_await.ipynb` — 异步编程（Agent 开发必备）
- `02_decorators_generators.ipynb` — 装饰器与生成器
- `03_type_hints_pydantic.ipynb` — 类型注解与 Pydantic（LLM 结构化输出基础）